# L4 · Taxonomy bệnh nhân — huấn luyện KMeans (bản debug/Colab)

Notebook này **mirror** script nguồn `scripts/l4/train_taxonomy.py` để debug trực quan / chạy trên Colab.
Script `.py` vẫn là **nguồn chân lý** (tái lập pickle). Khi sửa logic, sửa ở `.py` rồi đồng bộ lại đây.

**Cách làm & vì sao:** không có dữ liệu patient–workload có nhãn (McGill `no_observed_patient_samples`).
Nên taxonomy KHÔNG phải bài toán supervised. Ta sinh một *population synthetic* rồi dùng **KMeans 1D**
trên `workload_score = acuity + adl + isolation` để *khám phá điểm cắt tier tự nhiên* (kiểu Jenks),
đơn điệu theo cấu trúc. **Không phải deep learning** (dữ liệu bé, tabular).

> Honesty: population là SYNTHETIC, không phải bệnh nhân thật, không phải pain evidence. `workload_weight`
> là điểm tương đối, **không phải phút** (chưa có phép đổi điểm→phút được xác minh).

In [ ]:
# Colab: cài deps (bỏ qua nếu môi trường đã có). Chạy notebook từ thư mục gốc repo.
# %pip install 'scikit-learn>=1.5' 'numpy>=1.26' 'matplotlib>=3.8'
import os
# os.chdir('/content/K5-Day15-L3B-ToiYeuVingroup')  # chỉnh nếu chạy trên Colab
print('cwd:', os.getcwd())

In [ ]:
# 1) Sinh population synthetic (xác định theo seed) và 2) huấn luyện + xuất artifact.
!python scripts/l4/make_patient_population.py
!python scripts/l4/train_taxonomy.py

In [ ]:
# 3) Xem meta: điểm cắt tier KMeans học, silhouette theo k, phiên bản build.
import json
meta = json.load(open('codebase/L4/taxonomy_meta.json'))
print('k =', meta['k'], '| silhouette_max tại k =', meta['k_silhouette_max'])
print('tier_score_ranges =', meta['tier_score_ranges'])
print('fixture_agreement =', meta['fixture_agreement'])
print('build: sklearn', meta['sklearn_version'], '| numpy', meta['numpy_version'])

In [ ]:
# 4) Hình: silhouette theo k + histogram điểm cắt tier.
from IPython.display import Image
Image('docs/figures/l4_patient_clusters.png')

In [ ]:
# 5) Kiểm runtime: classify_patient dùng model đã ship, khớp ranges và fixture.
import sys; sys.path.insert(0, 'codebase')
from baseline import classify_patient
for a, d, i in [(1, 1, 0), (3, 2, 0), (5, 4, 1), (2, 2, 0), (4, 3, 0)]:
    r = classify_patient({'acuity_score': a, 'adl_dependency': d, 'isolation_precaution': bool(i)})
    print(f'a{a} d{d} i{i} score{a + d + i}: tier={r["tier"]} weight={r["workload_weight"]}')

## Đọc kết quả

- KMeans dời ranh giới low/med từ ngưỡng tay cũ (`3`) sang **`4`** — điểm cắt do dữ liệu quyết, không chọn tay.
- Silhouette tăng theo `k` vì điểm rời rạc (nhiều cluster luôn 'chặt' hơn); ta chọn `k=3` theo quy ước lâm sàng
  low/med/high và ghi `k_silhouette_max` để minh bạch.
- Artifact ship: `codebase/L4/taxonomy_model.pkl` + `taxonomy_meta.json`. Runtime `classify_patient` nạp model;
  thiếu sklearn/pickle thì fallback về đúng điểm cắt đã học (stdlib) — hành vi không đổi.

**Git hygiene:** commit notebook với output đã xoá (Kernel → Restart & Clear Output).